<a href="https://colab.research.google.com/github/krish26/DeepLearning/blob/main/catsanddogs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt

# =========================
# 1️⃣ Load Dataset
# =========================

(dataset, dataset_info) = tfds.load(
    'cats_vs_dogs',
    split='train',
    as_supervised=True,
    with_info=True
)

In [ ]:
print(dataset_info)

In [ ]:
print("Number of samples:", dataset_info.splits['train'].num_examples)
print("Number of classes:", dataset_info.features['label'].num_classes)
print("Class names:", dataset_info.features['label'].names)

In [ ]:
for image, label in dataset.take(1):
    print("Image shape:", image.shape)
    print("Label:", label.numpy())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,10))

for i, (image, label) in enumerate(dataset.take(9)):
    plt.subplot(3,3,i+1)
    plt.imshow(image)
    plt.title(dataset_info.features['label'].names[label])
    plt.axis("off")

plt.show()

In [ ]:
sizes = []

for image, _ in dataset.take(1000):
    sizes.append(image.shape[:2])

print("Sample sizes:", sizes[:10])

In [ ]:
for image, _ in dataset.take(1):
    print("Min pixel:", tf.reduce_min(image).numpy())
    print("Max pixel:", tf.reduce_max(image).numpy())

In [ ]:
bad_images = 0

for image, _ in dataset:
    if image.shape[-1] != 3:
        bad_images += 1

print("Non-RGB images:", bad_images)

In [ ]:
print(dataset_info.features)

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 32

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image / 255.0  # normalize
    return image, label

dataset = dataset.map(preprocess)

In [ ]:
dataset = dataset.shuffle(1000)

train_size = int(0.8 * dataset_info.splits['train'].num_examples)

train_dataset = dataset.take(train_size)
val_dataset = dataset.skip(train_size)

train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
model = tf.keras.Sequential([

    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(1, activation='sigmoid')  # Binary classification
])

model.summary()


In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10
)

In [ ]:
plt.plot(history.history['accuracy'], label='train_accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.legend()
plt.show()